In [1]:
import torch
import numpy as np

from train_1d_classifier_fixed import Downsample1DCNN, periods  # or paste model class here

device = torch.device("mps" if torch.backends.mps.is_available()
                      else ("cuda" if torch.cuda.is_available() else "cpu"))

checkpoint = torch.load("one_d_cnn_model.pth", map_location=device)

model = Downsample1DCNN(n_classes=len(periods)).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

idx_to_period = checkpoint["idx_to_period"]
print("Model loaded. Class index → period mapping:", idx_to_period)


Model loaded. Class index → period mapping: {0: 1000, 1: 2000, 2: 5000, 3: 7000, 4: 10000}


In [2]:
def predict_period(signal_array, model, device, idx_to_period):
    """
    signal_array: np.array of shape (L,) where L = sample_size (1e6 in your dataset)
    Returns predicted period value, e.g. 1000 or 5000.
    """
    arr = signal_array.astype('float32')
    # Same normalization used in the dataset
    arr = (arr - arr.mean()) / (arr.std() + 1e-6)

    x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)  # shape [1,1,L]

    with torch.no_grad():
        logits = model(x)
        pred_idx = int(logits.argmax(dim=1).item())

    return idx_to_period[pred_idx], pred_idx


In [5]:
import h5py

file = '/Users/straniero/Documents/Dphil/data_SPS/SPS.BQHT_MD2_20181026_131253.h5'

with h5py.File(file, 'r') as f:
    vert_delta = f['vertical/delta'][:]   
    horiz_delta = f['horizontal/delta'][:]
    horiz_sigma = f['horizontal/sigma'][:]
    vert_sigma = f['vertical/sigma'][:]

In [6]:
signal_np = vert_delta[:int(1e6)]  # or vert_delta, horiz_sigma, vert_sigma

pred_period, pred_idx = predict_period(signal_np, model, device, idx_to_period)


print("Predicted index:", pred_idx, "→ period", pred_period)

Predicted index: 0 → period 1000
